# Anomaly Detection

A comprehensive guide to detecting outliers and anomalies in data.

## Learning Objectives

- Understand types of anomalies
- Master statistical methods (Z-score, IQR)
- Learn Isolation Forest algorithm
- Implement Local Outlier Factor (LOF)
- Apply One-Class SVM for novelty detection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Types of Anomalies

**Point Anomalies:** Single data point is anomalous
**Contextual Anomalies:** Anomalous in specific context (e.g., temperature in winter)
**Collective Anomalies:** A group of data points is anomalous

**Use Cases:**
- Fraud detection
- Network intrusion detection
- Manufacturing defect detection
- Medical diagnosis

In [ ]:
# Generate synthetic dataset with anomalies
def generate_data_with_anomalies(n_samples=500, contamination=0.05):
    # Normal data
    n_normal = int(n_samples * (1 - contamination))
    n_anomaly = n_samples - n_normal
    
    # Generate normal points (two clusters)
    X_normal, _ = make_blobs(n_samples=n_normal, centers=[[2, 2], [-2, -2]], 
                             cluster_std=0.5, random_state=42)
    
    # Generate anomalies (random points)
    X_anomaly = np.random.uniform(low=-6, high=6, size=(n_anomaly, 2))
    
    # Combine
    X = np.vstack([X_normal, X_anomaly])
    y = np.hstack([np.ones(n_normal), -np.ones(n_anomaly)])  # 1 = normal, -1 = anomaly
    
    # Shuffle
    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

X, y_true = generate_data_with_anomalies(n_samples=500, contamination=0.1)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X[y_true == 1, 0], X[y_true == 1, 1], c='blue', label='Normal', alpha=0.6, s=30)
plt.scatter(X[y_true == -1, 0], X[y_true == -1, 1], c='red', label='Anomaly', alpha=0.8, s=50, marker='x')
plt.legend()
plt.title('Synthetic Dataset with Anomalies')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

print(f"Total samples: {len(X)}")
print(f"Normal: {(y_true == 1).sum()}, Anomalies: {(y_true == -1).sum()}")

## 2. Statistical Methods

### Z-Score Method
Points with |z| > threshold (typically 3) are considered anomalies.

### IQR Method
Points outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR] are anomalies.

In [ ]:
# Z-score method
def zscore_anomaly_detection(X, threshold=3):
    z_scores = np.abs(stats.zscore(X))
    is_anomaly = np.any(z_scores > threshold, axis=1)
    return np.where(is_anomaly, -1, 1)

# IQR method
def iqr_anomaly_detection(X, k=1.5):
    Q1 = np.percentile(X, 25, axis=0)
    Q3 = np.percentile(X, 75, axis=0)
    IQR = Q3 - Q1
    
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR
    
    is_anomaly = np.any((X < lower) | (X > upper), axis=1)
    return np.where(is_anomaly, -1, 1)

# Apply methods
y_zscore = zscore_anomaly_detection(X)
y_iqr = iqr_anomaly_detection(X)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Ground truth
axes[0].scatter(X[:, 0], X[:, 1], c=y_true, cmap='coolwarm', s=20)
axes[0].set_title('Ground Truth')

# Z-score
axes[1].scatter(X[:, 0], X[:, 1], c=y_zscore, cmap='coolwarm', s=20)
axes[1].set_title(f'Z-Score (Detected: {(y_zscore == -1).sum()})')

# IQR
axes[2].scatter(X[:, 0], X[:, 1], c=y_iqr, cmap='coolwarm', s=20)
axes[2].set_title(f'IQR Method (Detected: {(y_iqr == -1).sum()})')

plt.tight_layout()
plt.show()

## 3. Isolation Forest

**Key Idea:** Anomalies are easier to isolate (require fewer splits).

**Algorithm:**
1. Build random trees by randomly selecting features and split values
2. Anomalies have shorter path lengths (isolated quickly)
3. Score = average path length across all trees

In [ ]:
# Isolation Forest
iso_forest = IsolationForest(contamination=0.1, random_state=42, n_estimators=100)
y_iforest = iso_forest.fit_predict(X)

# Get anomaly scores
scores_iforest = iso_forest.decision_function(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predictions
colors = ['blue' if label == 1 else 'red' for label in y_iforest]
axes[0].scatter(X[:, 0], X[:, 1], c=colors, s=20, alpha=0.7)
axes[0].set_title(f'Isolation Forest Predictions\n(Detected: {(y_iforest == -1).sum()} anomalies)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Anomaly scores
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=scores_iforest, cmap='RdYlBu', s=20)
plt.colorbar(scatter, ax=axes[1], label='Anomaly Score')
axes[1].set_title('Isolation Forest Anomaly Scores\n(Lower = More Anomalous)')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize decision boundary
xx, yy = np.meshgrid(np.linspace(-8, 8, 100), np.linspace(-8, 8, 100))
Z = iso_forest.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, Z, levels=20, cmap='RdYlBu')
plt.colorbar(label='Anomaly Score')
plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')
plt.scatter(X[y_true == 1, 0], X[y_true == 1, 1], c='blue', s=20, alpha=0.5, label='Normal')
plt.scatter(X[y_true == -1, 0], X[y_true == -1, 1], c='red', s=50, marker='x', label='Anomaly')
plt.legend()
plt.title('Isolation Forest Decision Boundary')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

## 4. Local Outlier Factor (LOF)

**Key Idea:** Compare local density of a point to its neighbors.

**Algorithm:**
1. Calculate k-distance for each point
2. Calculate local reachability density (LRD)
3. LOF = ratio of LRD of neighbors to point's LRD
4. LOF > 1 indicates anomaly

In [ ]:
# Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
y_lof = lof.fit_predict(X)

# Get LOF scores (negative_outlier_factor_ is negative, more negative = more anomalous)
lof_scores = lof.negative_outlier_factor_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predictions
colors = ['blue' if label == 1 else 'red' for label in y_lof]
axes[0].scatter(X[:, 0], X[:, 1], c=colors, s=20, alpha=0.7)
axes[0].set_title(f'LOF Predictions\n(Detected: {(y_lof == -1).sum()} anomalies)')

# LOF scores
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=lof_scores, cmap='RdYlBu', s=20)
plt.colorbar(scatter, ax=axes[1], label='LOF Score')
axes[1].set_title('LOF Scores\n(More Negative = More Anomalous)')

plt.tight_layout()
plt.show()

In [ ]:
# Effect of n_neighbors
n_neighbors_list = [5, 10, 20, 50]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, n in zip(axes, n_neighbors_list):
    lof = LocalOutlierFactor(n_neighbors=n, contamination=0.1)
    y_pred = lof.fit_predict(X)
    
    colors = ['blue' if label == 1 else 'red' for label in y_pred]
    ax.scatter(X[:, 0], X[:, 1], c=colors, s=15, alpha=0.7)
    ax.set_title(f'n_neighbors={n}\nDetected: {(y_pred == -1).sum()}')

plt.suptitle('Effect of n_neighbors on LOF', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. One-Class SVM

**Key Idea:** Learn a boundary around normal data.

**Algorithm:**
1. Map data to high-dimensional space using kernel
2. Find hyperplane that separates data from origin
3. Points outside the hyperplane are anomalies

In [ ]:
# Scale data for SVM
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# One-Class SVM
ocsvm = OneClassSVM(nu=0.1, kernel='rbf', gamma='auto')
y_ocsvm = ocsvm.fit_predict(X_scaled)

# Decision function
scores_ocsvm = ocsvm.decision_function(X_scaled)

# Visualize decision boundary
xx, yy = np.meshgrid(np.linspace(-4, 4, 100), np.linspace(-4, 4, 100))
Z = ocsvm.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, Z, levels=20, cmap='RdYlBu')
plt.colorbar(label='Decision Function')
plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')
plt.scatter(X_scaled[y_true == 1, 0], X_scaled[y_true == 1, 1], c='blue', s=20, alpha=0.5, label='Normal')
plt.scatter(X_scaled[y_true == -1, 0], X_scaled[y_true == -1, 1], c='red', s=50, marker='x', label='Anomaly')
plt.legend()
plt.title(f'One-Class SVM\n(Detected: {(y_ocsvm == -1).sum()} anomalies)')
plt.xlabel('Feature 1 (scaled)')
plt.ylabel('Feature 2 (scaled)')
plt.show()

## 6. Method Comparison

In [ ]:
# Compare all methods
methods = {
    'Z-Score': zscore_anomaly_detection(X),
    'IQR': iqr_anomaly_detection(X),
    'Isolation Forest': iso_forest.fit_predict(X),
    'LOF': LocalOutlierFactor(contamination=0.1).fit_predict(X),
    'One-Class SVM': OneClassSVM(nu=0.1).fit_predict(X_scaled)
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

# Ground truth
colors_true = ['blue' if y == 1 else 'red' for y in y_true]
axes[0].scatter(X[:, 0], X[:, 1], c=colors_true, s=20, alpha=0.7)
axes[0].set_title('Ground Truth')

for ax, (name, y_pred) in zip(axes[1:], methods.items()):
    colors = ['blue' if y == 1 else 'red' for y in y_pred]
    ax.scatter(X[:, 0], X[:, 1], c=colors, s=20, alpha=0.7)
    ax.set_title(f'{name}\nDetected: {(y_pred == -1).sum()}')

plt.tight_layout()
plt.show()

In [ ]:
# Evaluation metrics
def evaluate_anomaly_detection(y_true, y_pred, method_name):
    # Convert to binary: anomaly = 1 (positive class)
    y_true_binary = (y_true == -1).astype(int)
    y_pred_binary = (y_pred == -1).astype(int)
    
    from sklearn.metrics import precision_score, recall_score, f1_score
    
    precision = precision_score(y_true_binary, y_pred_binary)
    recall = recall_score(y_true_binary, y_pred_binary)
    f1 = f1_score(y_true_binary, y_pred_binary)
    
    return {
        'Method': method_name,
        'Precision': precision,
        'Recall': recall,
        'F1': f1
    }

results = [evaluate_anomaly_detection(y_true, y_pred, name) 
           for name, y_pred in methods.items()]

results_df = pd.DataFrame(results)
print("Anomaly Detection Method Comparison:")
print(results_df.to_string(index=False))

## 7. Practical Example: Credit Card Fraud Detection

In [ ]:
# Simulate credit card transaction data
np.random.seed(42)
n_transactions = 10000
n_fraud = 100  # 1% fraud rate

# Normal transactions
normal = pd.DataFrame({
    'amount': np.random.lognormal(3, 1, n_transactions - n_fraud),
    'time_since_last': np.random.exponential(10, n_transactions - n_fraud),
    'distance_from_home': np.random.exponential(5, n_transactions - n_fraud),
    'is_fraud': 0
})

# Fraudulent transactions (different patterns)
fraud = pd.DataFrame({
    'amount': np.random.lognormal(5, 1.5, n_fraud),  # Higher amounts
    'time_since_last': np.random.exponential(1, n_fraud),  # Quick succession
    'distance_from_home': np.random.exponential(50, n_fraud),  # Far from home
    'is_fraud': 1
})

# Combine
transactions = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)

print(f"Dataset shape: {transactions.shape}")
print(f"Fraud rate: {transactions['is_fraud'].mean()*100:.2f}%")
print("\nFeature statistics:")
print(transactions.groupby('is_fraud').mean())

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['amount', 'time_since_last', 'distance_from_home']):
    transactions[transactions['is_fraud'] == 0][col].hist(ax=ax, bins=50, alpha=0.5, label='Normal')
    transactions[transactions['is_fraud'] == 1][col].hist(ax=ax, bins=50, alpha=0.5, label='Fraud')
    ax.set_title(col)
    ax.legend()
    ax.set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Prepare data
X_fraud = transactions[['amount', 'time_since_last', 'distance_from_home']].values
y_fraud = transactions['is_fraud'].values

# Scale
scaler = StandardScaler()
X_fraud_scaled = scaler.fit_transform(X_fraud)

# Apply anomaly detection
contamination = 0.01  # Expected fraud rate

models = {
    'Isolation Forest': IsolationForest(contamination=contamination, random_state=42),
    'One-Class SVM': OneClassSVM(nu=contamination)
}

results = []
for name, model in models.items():
    y_pred = model.fit_predict(X_fraud_scaled)
    y_pred_binary = (y_pred == -1).astype(int)
    
    from sklearn.metrics import precision_score, recall_score, f1_score
    
    results.append({
        'Model': name,
        'Precision': precision_score(y_fraud, y_pred_binary),
        'Recall': recall_score(y_fraud, y_pred_binary),
        'F1': f1_score(y_fraud, y_pred_binary),
        'Detected': y_pred_binary.sum()
    })

results_df = pd.DataFrame(results)
print("Fraud Detection Results:")
print(results_df.to_string(index=False))

## 8. Autoencoder for Anomaly Detection

In [ ]:
# Simple autoencoder concept (reconstruction error)
from sklearn.neural_network import MLPRegressor

# Train autoencoder on normal data only
X_normal = X_fraud_scaled[y_fraud == 0]

# Simple reconstruction model (input -> compressed -> output)
# Using sklearn's MLP as a simple approximation
autoencoder = MLPRegressor(
    hidden_layer_sizes=(16, 3, 16),  # Bottleneck of 3
    activation='relu',
    max_iter=500,
    random_state=42
)

# Train on normal data (reconstruct itself)
autoencoder.fit(X_normal, X_normal)

# Calculate reconstruction error for all data
X_reconstructed = autoencoder.predict(X_fraud_scaled)
reconstruction_error = np.mean((X_fraud_scaled - X_reconstructed) ** 2, axis=1)

# Use percentile threshold
threshold = np.percentile(reconstruction_error, 99)
y_ae_pred = (reconstruction_error > threshold).astype(int)

print(f"Threshold (99th percentile): {threshold:.4f}")
print(f"Detected anomalies: {y_ae_pred.sum()}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reconstruction error distribution
axes[0].hist(reconstruction_error[y_fraud == 0], bins=50, alpha=0.5, label='Normal', density=True)
axes[0].hist(reconstruction_error[y_fraud == 1], bins=50, alpha=0.5, label='Fraud', density=True)
axes[0].axvline(threshold, color='r', linestyle='--', label=f'Threshold={threshold:.3f}')
axes[0].set_xlabel('Reconstruction Error')
axes[0].set_ylabel('Density')
axes[0].set_title('Reconstruction Error Distribution')
axes[0].legend()

# Error by class
axes[1].boxplot([reconstruction_error[y_fraud == 0], reconstruction_error[y_fraud == 1]], 
                labels=['Normal', 'Fraud'])
axes[1].set_ylabel('Reconstruction Error')
axes[1].set_title('Reconstruction Error by Class')

plt.tight_layout()
plt.show()

## 9. Algorithm Selection Guide

| Method | Best For | Complexity | Pros | Cons |
|--------|----------|------------|------|------|
| Z-Score/IQR | Univariate, Gaussian | O(n) | Simple, fast | Assumes distribution |
| Isolation Forest | High-dimensional | O(n log n) | Fast, scalable | Struggles with local anomalies |
| LOF | Local anomalies | O(n²) | Captures local density | Slow for large data |
| One-Class SVM | Non-linear boundaries | O(n³) | Flexible kernel | Sensitive to parameters |
| Autoencoder | Complex patterns | Varies | Learns representations | Requires tuning |

In [ ]:
# Summary table
summary = pd.DataFrame({
    'Method': ['Z-Score', 'IQR', 'Isolation Forest', 'LOF', 'One-Class SVM', 'Autoencoder'],
    'Type': ['Statistical', 'Statistical', 'Ensemble', 'Density', 'Kernel', 'Neural'],
    'Handles Multivariate': ['Partially', 'Partially', 'Yes', 'Yes', 'Yes', 'Yes'],
    'Scalability': ['High', 'High', 'High', 'Low', 'Low', 'Medium'],
    'Key Parameter': ['threshold', 'k', 'contamination', 'n_neighbors', 'nu, kernel', 'architecture']
})

print("Anomaly Detection Methods Summary:")
print(summary.to_string(index=False))

## 10. Key Takeaways

1. **Anomaly detection is unsupervised** - we often don't have labeled anomalies
2. **Statistical methods (Z-score, IQR)** work well for simple, low-dimensional data
3. **Isolation Forest** is fast and effective for high-dimensional data
4. **LOF** is best for detecting local anomalies in varying density
5. **One-Class SVM** works well with proper kernel selection
6. **Autoencoders** can learn complex patterns but require more tuning
7. **Contamination parameter** is crucial - estimate the expected anomaly rate
8. **Always scale features** before using distance-based methods